# 13.2 偏好資料怎麼表示？


一篇回答1+1得2，另一篇回答2+2得3，前者看似正確、後者錯誤，卻不是同一道問題的比較。訓練需要「同一提問，兩種續寫」，才知道應該提高哪一種回答的相對偏好。本節將它整理成prompt、chosen、rejected三件組，並看轉換後的答案位置。

前置是[13.1偏好條件](https://birdhackor.github.io/tiny-perceptron-vlm/13.1.html)與[7.3答案遮罩](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)。chosen指較佳回答，rejected指較差回答；兩個名稱是人給的標籤。`render_chat`會將角色、提問與回答轉成下一位置預測所需的輸入與目標，問題本身不計回答分數，但仍是模型讀得到的上下文。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import render_chat

pair = {"prompt": "1+1=?", "chosen": "2", "rejected": "3"}
for key in ["chosen", "rejected"]:
    x, y = render_chat(
        [
            {"role": "user", "content": pair["prompt"]},
            {"role": "assistant", "content": pair[key]},
        ]
    )
    print(key, "輸入", x.tolist(), "目標", y.tolist())

輸入僅有一個共享prompt，兩次只改assistant的文字。預設ByteTokenizer將UTF-8每個位元組加8作為普通編號，特定編號保留給角色：1是開頭、3用戶、4助手、2結束。英文數字2與3編碼為58與59，因此chosen輸入為`[1,3,57,51,57,69,71,2,4,58]`，rejected最後一格則59。前九個輸入位置完全相同。

兩份目標都是前八項-100，之後為回答編號58或59，再加結束2，共10項。-100表示不算這個位置的猜錯代價；回答第一格由前面的助手角色預測，最後由回答預測結束。這裡已shift，不能在後續算分時再挪一次，否則會比較錯誤位置。

這個對應也解釋為什麼要保留共同上下文。若rejected用另一題，分數差可能來自問題難度，而非同一條件下回答的好壞。資料來源、問題身份與偏好理由可另外保存，便於重複去除與授權檢查；核心三件組仍必須保持共享prompt。

對話輸入與目標長度同為10，但角色編號位置並非各自對應同樣數字的目標。助手編號4在輸入索引8，對應目標58，表示開始生成2；輸入最後58對應目標2，表示回答結束。這個具體配對說明為什麼目標前八項忽略而不是前九項。兩條回答的公共問題區域可共享格式處理，評分時卻要分別用自己的答案前文計算，不能拿chosen的輸入搭rejected標籤。條件機率就是先給定已看到的前文，再問接下來某個編號有多少機會。例如評rejected的結束編號2時，前文最後應是59，也就是剛回答了3；若套chosen的輸入，前文最後卻是58，算到的是回答2之後結束的機率。混搭後的分數便不再代表rejected這篇完整回答。

練習只把rejected改4，先預測其普通編號從59變60，chosen所有數字保持不變，再執行核對。接著說明為什麼不能只給rejected換題目：那會破壞這組三件組的比較條件。真訓練時兩條續寫長度可以不同，但各自標籤必須對齊自己的完整回答。
